In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import DistilBertModel, DistilBertTokenizer
from transformers import CLIPModel
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from PIL import Image
import timm
from peft import LoraConfig, get_peft_model

In [2]:
df = pd.read_pickle('dish_dataset.pkl')
df.head()

,dish_id,total_calories,total_mass,total_fat,total_carb,total_protein,ingredients,rgb_image_path
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,"soy sauce, garlic, white rice, parsley, onions...",data/overhead/dish_1561662216/rgb.png
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,"pepper, white rice, mixed greens, garlic, soy ...",data/overhead/dish_1561662054/rgb.png
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,"jalapenos, lemon juice, pork, wheat berry, cab...",data/overhead/dish_1562008979/rgb.png
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,"cherry tomatoes, cucumbers, baby carrots",data/overhead/dish_1560455030/rgb.png
5,dish_1558372433,74.360001,143.0,0.286000,0.429000,20.020000,deprecated,data/overhead/dish_1558372433/rgb.png


In [3]:
class DishDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['rgb_image_path']
        image = Image.open(img_path).convert('RGB').copy()
        if self.transform:
            image = self.transform(image)
        
        # Targets: total_calories, total_mass, total_fat, total_carb, total_protein
        targets = torch.tensor([
            row['total_calories'],
            row['total_mass'],
            row['total_fat'],
            row['total_carb'],
            row['total_protein']
        ], dtype=torch.float32)
        
        return image, targets

# Define transforms
train_transform = transforms.Compose([
    
    transforms.RandomResizedCrop(
        224,
        scale=(0.6, 1.0),
        ratio=(0.9, 1.1)
    ),

    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),

    transforms.RandomAffine(
        degrees=20,
        translate=(0.05, 0.05),
        scale=(0.9, 1.1)
    ),

    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.25,
        hue=0.05
    ),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 1.5)
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.08)
    ),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Split data
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Create datasets
train_dataset = DishDataset(train_df, transform=train_transform)
val_dataset = DishDataset(val_df, transform=val_transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Increased from 16
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [4]:
class DeiTRegressor(nn.Module):
    def __init__(self, n_outputs):
        super().__init__()
        

        self.backbone = timm.create_model(
            "deit_tiny_patch16_224",
            pretrained=True,
            num_classes=0
        )
        
        embed_dim = self.backbone.num_features  # 192
        
        # Custom regression head - reduced size to prevent overfitting
        self.regression_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128),  # Reduced from 256
            nn.GELU(),
            nn.Dropout(0.5),  # Increased from 0.2
            nn.Linear(128, n_outputs)
        )
    
    def forward(self, x):
        features = self.backbone(x)  # [B, 192]
        output = self.regression_head(features)  # [B, n_outputs]
        return output

In [5]:
def build_frozen_model(n_outputs, device):
    model = DeiTRegressor(n_outputs).to(device)
    
    # Freeze backbone
    for param in model.backbone.parameters():
        param.requires_grad = False
    
    optimizer = torch.optim.AdamW(
        model.regression_head.parameters(),
        lr=1e-4, 
        weight_decay=5e-4
    )
    
    return model, optimizer

def build_full_finetune_model(n_outputs, device):
    model = DeiTRegressor(n_outputs).to(device)
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-5,  # Reduced from 1e-4 for slower, more stable learning
        weight_decay=5e-4  # Increased from 1e-4 for stronger regularization
    )
    
    return model, optimizer

def build_lora_model(n_outputs, device):
    model = DeiTRegressor(n_outputs).to(device)
    
    # Freeze backbone first
    for param in model.backbone.parameters():
        param.requires_grad = False
    
    # Configure LoRA with reduced capacity
    lora_config = LoraConfig(
        r=4,  # Reduced from 8 to limit adapters
        lora_alpha=8,  # Reduced from 16 (alpha=r is less aggressive)
        target_modules=["qkv"],  # attention projections
        lora_dropout=0.3,  # Increased from 0.1 for better regularization
        bias="none"
    )
    
    # Inject LoRA into backbone
    model.backbone = get_peft_model(model.backbone, lora_config)
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=2e-4,  # Reduced from 5e-4 for slower adaptation
        weight_decay=5e-4  # Increased from 1e-4 for stronger regularization
    )
    
    return model, optimizer

In [6]:
def train(model, optimizer, train_loader, val_loader, device, epochs, normalize_targets=True, patience=5):
    """
    Training loop with detailed loss reporting and early stopping.
    
    Args:
        normalize_targets: If True, normalizes each target independently
        patience: Stop training if val loss doesn't improve for this many epochs
    """
    output_names = ['total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein']
    criterion = nn.SmoothL1Loss(reduction='none')  # Per-element loss
    
    # Compute normalization factors from training data
    if normalize_targets:
        targets_all = torch.cat([targets for _, targets in train_loader], dim=0)
        targets_mean = targets_all.mean(dim=0)
        targets_std = targets_all.std(dim=0) + 1e-8  # avoid division by zero
        targets_mean = targets_mean.to(device)
        targets_std = targets_std.to(device)
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        # ===== TRAINING =====
        model.train()
        losses_per_output = [0.0] * 5  # Loss for each of 5 outputs
        
        for images, targets in train_loader:
            images = images.to(device)
            targets = targets.to(device).float()
            
            # Normalize targets for balanced loss
            if normalize_targets:
                targets_norm = (targets - targets_mean) / targets_std
            else:
                targets_norm = targets
            
            optimizer.zero_grad()
            outputs = model(images)  # [B, 5]
            
            # Compute per-output loss (in normalized space)
            loss_per_output = criterion(outputs, targets_norm).mean(dim=0)  # [5]
            
            # Denormalize predictions to show actual errors
            outputs_denorm = outputs * targets_std + targets_mean
            mae_per_output = torch.abs(outputs_denorm - targets).mean(dim=0)  # [5] MAE in original units
            
            # Total loss (average across 5 outputs)
            loss = loss_per_output.mean()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            # Track loss per output
            for i in range(5):
                losses_per_output[i] += loss_per_output[i].item()
        
        avg_losses = [l / len(train_loader) for l in losses_per_output]
        avg_train_loss = sum(avg_losses) / 5
        
        # ===== VALIDATION =====
        model.eval()
        val_losses_per_output = [0.0] * 5
        val_mae_per_output = [0.0] * 5
        
        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device)
                targets = targets.to(device).float()
                
                if normalize_targets:
                    targets_norm = (targets - targets_mean) / targets_std
                else:
                    targets_norm = targets
                
                outputs = model(images)
                loss_per_output = criterion(outputs, targets_norm).mean(dim=0)
                outputs_denorm = outputs * targets_std + targets_mean
                mae_per_output = torch.abs(outputs_denorm - targets).mean(dim=0)
                
                for i in range(5):
                    val_losses_per_output[i] += loss_per_output[i].item()
                    val_mae_per_output[i] += mae_per_output[i].item()
        
        val_avg_losses = [l / len(val_loader) for l in val_losses_per_output]
        val_avg_mae = [l / len(val_loader) for l in val_mae_per_output]
        val_avg_loss = sum(val_avg_losses) / 5
        
        # Early stopping: check if validation loss improved
        overfitting_gap = val_avg_loss - avg_train_loss
        if val_avg_loss < best_val_loss:
            best_val_loss = val_avg_loss
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Print epoch results with per-output breakdown
        gap_indicator = " ⚠️ OVERFITTING" if overfitting_gap > 0.03 else ""
        print(f"\nEpoch {epoch+1}/{epochs}{gap_indicator} [Patience: {patience_counter}/{patience}]")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_avg_loss:.4f} | Gap: {overfitting_gap:.4f}")
        print("Val MAE per output:")
        for i, name in enumerate(output_names):
            print(f"  {name}: {val_avg_mae[i]:.4f}", end="  ")
        print()
        
        # Early stopping
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping triggered after {epoch+1} epochs (patience={patience})")
            break

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
n_outputs = 5

# 1️⃣ Frozen
model_frozen, opt_frozen = build_frozen_model(n_outputs, device)

# 2️⃣ Full fine-tune
model_full, opt_full = build_full_finetune_model(n_outputs, device)

# 3️⃣ LoRA
model_lora, opt_lora = build_lora_model(n_outputs, device)

In [8]:
train(model_frozen, opt_frozen, train_loader, val_loader, device, epochs=10)


Epoch 1/10 [Patience: 0/5]
Train Loss: 0.3442 | Val Loss: 0.2885 | Gap: -0.0556
Val MAE per output:
  total_calories: 151.9267    total_mass: 99.0940    total_fat: 8.9961    total_carb: 11.5768    total_protein: 12.6657  

Epoch 2/10 [Patience: 0/5]
Train Loss: 0.2824 | Val Loss: 0.2462 | Gap: -0.0362
Val MAE per output:
  total_calories: 133.9861    total_mass: 81.1959    total_fat: 8.2724    total_carb: 10.7631    total_protein: 11.1854  

Epoch 3/10 [Patience: 0/5]
Train Loss: 0.2521 | Val Loss: 0.2263 | Gap: -0.0258
Val MAE per output:
  total_calories: 124.4805    total_mass: 73.5949    total_fat: 8.0982    total_carb: 10.2527    total_protein: 10.5960  

Epoch 4/10 [Patience: 0/5]
Train Loss: 0.2382 | Val Loss: 0.2168 | Gap: -0.0215
Val MAE per output:
  total_calories: 118.4948    total_mass: 69.6406    total_fat: 7.8261    total_carb: 9.7993    total_protein: 10.1894  

Epoch 5/10 [Patience: 0/5]
Train Loss: 0.2260 | Val Loss: 0.2071 | Gap: -0.0188
Val MAE per output:
  total_

In [9]:
train(model_full, opt_full, train_loader, val_loader, device, epochs=10)


Epoch 1/10 [Patience: 0/5]
Train Loss: 0.2300 | Val Loss: 0.1723 | Gap: -0.0576
Val MAE per output:
  total_calories: 89.4367    total_mass: 65.5088    total_fat: 6.7437    total_carb: 9.1805    total_protein: 8.0425  

Epoch 2/10 [Patience: 0/5]
Train Loss: 0.1734 | Val Loss: 0.1682 | Gap: -0.0052
Val MAE per output:
  total_calories: 84.0570    total_mass: 65.6698    total_fat: 6.2294    total_carb: 8.6003    total_protein: 7.9123  

Epoch 3/10 [Patience: 0/5]
Train Loss: 0.1528 | Val Loss: 0.1254 | Gap: -0.0275
Val MAE per output:
  total_calories: 69.3611    total_mass: 54.7875    total_fat: 5.7083    total_carb: 7.5650    total_protein: 6.7192  

Epoch 4/10 [Patience: 0/5]
Train Loss: 0.1423 | Val Loss: 0.1241 | Gap: -0.0181
Val MAE per output:
  total_calories: 69.1031    total_mass: 52.6909    total_fat: 5.7153    total_carb: 7.7119    total_protein: 6.2462  

Epoch 5/10 [Patience: 0/5]
Train Loss: 0.1345 | Val Loss: 0.1139 | Gap: -0.0206
Val MAE per output:
  total_calories: 7

In [10]:
train(model_lora, opt_lora, train_loader, val_loader, device, epochs=10)


Epoch 1/10 [Patience: 0/5]
Train Loss: 0.2897 | Val Loss: 0.2153 | Gap: -0.0744
Val MAE per output:
  total_calories: 116.9825    total_mass: 67.8096    total_fat: 7.8295    total_carb: 10.5102    total_protein: 9.6468  

Epoch 2/10 [Patience: 0/5]
Train Loss: 0.2139 | Val Loss: 0.1826 | Gap: -0.0313
Val MAE per output:
  total_calories: 98.3151    total_mass: 63.1881    total_fat: 6.9929    total_carb: 9.3252    total_protein: 8.5556  

Epoch 3/10 [Patience: 0/5]
Train Loss: 0.1900 | Val Loss: 0.1636 | Gap: -0.0264
Val MAE per output:
  total_calories: 89.2843    total_mass: 58.0747    total_fat: 6.8099    total_carb: 8.7293    total_protein: 8.1814  

Epoch 4/10 [Patience: 0/5]
Train Loss: 0.1773 | Val Loss: 0.1556 | Gap: -0.0217
Val MAE per output:
  total_calories: 85.2921    total_mass: 57.1057    total_fat: 6.4792    total_carb: 8.3974    total_protein: 7.9675  

Epoch 5/10 [Patience: 0/5]
Train Loss: 0.1669 | Val Loss: 0.1505 | Gap: -0.0164
Val MAE per output:
  total_calories:

### Multimodal architecture (with ingredients)

In [ ]:
class MultiModalNutritionModel(nn.Module):
    
    def __init__(self, n_outputs):
        super().__init__()

        # IMAGE BACKBONE
        self.image_encoder = timm.create_model(
            "deit_tiny_patch16_224",
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_encoder.num_features  # 192

        # TEXT BACKBONE
        self.text_encoder = DistilBertModel.from_pretrained(
            "distilbert-base-uncased"
        )

        text_dim = 768

        # Freeze most of the encoders
        for param in self.image_encoder.parameters():
            param.requires_grad = False

        for param in self.text_encoder.parameters():
            param.requires_grad = False

        # FUSION HEAD
        fusion_dim = image_dim + text_dim

        self.regressor = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, n_outputs)
        )

    def forward(self, image, input_ids, attention_mask):

        # Image features
        image_features = self.image_encoder(image)

        # Text features
        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        text_features = text_outputs.last_hidden_state[:,0,:]  # CLS token

        # Fusion
        combined = torch.cat([image_features, text_features], dim=1)

        outputs = self.regressor(combined)

        return outputs

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

encoding = tokenizer(
    ingredient_text,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)